In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
#from pyspark.sql.functions import dense_rank, desc, overfrom pyspark.sql.window import Window


In [0]:
df = spark.read.format("parquet")\
    .load("abfss://bronze@databricksuk2025.dfs.core.windows.net/orders")

In [0]:
# Drop column

df = df.drop("_rescued_data")
# df.display()

In [0]:
# Add column Year

df = df.withColumn("year", year(col("order_date")))

In [0]:
# DENSE_RANK, NUMBER_ROW, RANK FUNCTIONS 

#df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
#df1.display()

### Classes OOP

In [0]:
# Class to reuse the functions

class windows:

    def dense_rank(self,df):

        df_dense_rank = df.withColumn("Dense_rank",dense_rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_dense_rank
    
    def rank(self,df):
        df_rank = df.withColumn("Rank",rank().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_rank
    
    def row_number(self,df):
        df_row_number = df.withColumn("Row_Number",row_number().over(Window.partitionBy("year").orderBy(desc("total_amount"))))
        return df_row_number
        

In [0]:
# call the class
obj = windows()

In [0]:
df_new = obj.dense_rank(df)
df_new.display()

In [0]:
df_new = obj.rank(df)
df_new.display()

In [0]:
df_new = obj.row_number(df)
df_new.display()

### Data Writing

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@databricksuk2025.dfs.core.windows.net/orders")

In [0]:
%sql

-- CREATING THE SILVER TABLE
create table if not exists databricks_cata.silver.orders_silver 
using DELTA
Location 'abfss://silver@databricksuk2025.dfs.core.windows.net/orders'